In [1]:
# ============================================================
# ELEVANCE SKILLS - STACKED AREA CHART
# ============================================================

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import datetime, timezone, timedelta


# ============================================================
# 1. CSV FILE PATH
# ============================================================

file_path = r"C:\Users\Harshitha.M\Downloads\Play Store Data (1).csv"

df = pd.read_csv(file_path)

print("CSV loaded successfully!")
print("Total rows:", len(df))
print("Columns:", list(df.columns))


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

df.columns = df.columns.str.strip()


# ============================================================
# 3. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "App",
    "Category",
    "Rating",
    "Reviews",
    "Size",
    "Installs",
    "Last Updated"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:

    print("\nERROR - Missing columns:")
    print(missing_columns)

    print("\nAvailable columns:")
    print(list(df.columns))

else:

    # ========================================================
    # 4. CLEAN RATING
    # ========================================================

    df["Rating"] = pd.to_numeric(
        df["Rating"],
        errors="coerce"
    )

    # Average rating >= 4.2
    df = df[
        df["Rating"] >= 4.2
    ].copy()


    # ========================================================
    # 5. CLEAN APP NAME
    # ========================================================

    df["App"] = (
        df["App"]
        .astype(str)
        .str.strip()
    )

    # App name must NOT contain numbers
    df = df[
        ~df["App"].str.contains(
            r"\d",
            regex=True,
            na=False
        )
    ].copy()


    # ========================================================
    # 6. CATEGORY FILTER
    # ========================================================

    df["Category"] = (
        df["Category"]
        .astype(str)
        .str.strip()
    )

    # Category must start with T or P
    df = df[
        df["Category"]
        .str.upper()
        .str.startswith(("T", "P"))
    ].copy()


    # ========================================================
    # 7. CLEAN REVIEWS
    # ========================================================

    df["Reviews"] = pd.to_numeric(
        df["Reviews"],
        errors="coerce"
    )

    # Reviews > 1000
    df = df[
        df["Reviews"] > 1000
    ].copy()


    # ========================================================
    # 8. CLEAN SIZE
    # ========================================================

    def convert_size(value):

        if pd.isna(value):
            return np.nan

        value = str(value).strip()

        # Remove spaces
        value = value.replace(" ", "")

        # Variable size
        if value.lower() == "varieswithdevice":
            return np.nan

        try:

            if value.lower().endswith("m"):
                return float(
                    value[:-1]
                )

            elif value.lower().endswith("k"):
                return float(
                    value[:-1]
                ) / 1024

            else:
                return np.nan

        except:

            return np.nan


    df["Size_MB"] = df["Size"].apply(
        convert_size
    )

    # Size between 20 MB and 80 MB
    df = df[
        (df["Size_MB"] >= 20) &
        (df["Size_MB"] <= 80)
    ].copy()


    # ========================================================
    # 9. CLEAN INSTALLS
    # ========================================================

    def convert_installs(value):

        if pd.isna(value):
            return np.nan

        value = str(value).strip()

        value = value.replace("+", "")
        value = value.replace(",", "")

        try:
            return float(value)
        except:
            return np.nan


    df["Installs_Numeric"] = df["Installs"].apply(
        convert_installs
    )

    df = df.dropna(
        subset=["Installs_Numeric"]
    ).copy()


    # ========================================================
    # 10. CLEAN LAST UPDATED
    # ========================================================

    df["Last Updated"] = pd.to_datetime(
        df["Last Updated"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["Last Updated"]
    ).copy()


    # ========================================================
    # 11. CREATE MONTH
    # ========================================================

    df["Month"] = (
        df["Last Updated"]
        .dt.to_period("M")
        .dt.to_timestamp()
    )


    # ========================================================
    # 12. TRANSLATE CATEGORIES
    # ========================================================

    def translate_category(category):

        category = str(category).strip()

        if category.upper() == "TRAVEL & LOCAL":
            # French
            return "Voyage et local"

        elif category.upper() == "PRODUCTIVITY":
            # Spanish
            return "Productividad"

        elif category.upper() == "PHOTOGRAPHY":
            # Japanese
            return "写真"

        else:
            return category


    df["Category_Display"] = df["Category"].apply(
        translate_category
    )


    # ========================================================
    # 13. GROUP BY MONTH AND CATEGORY
    # ========================================================

    monthly_data = (
        df.groupby(
            [
                "Month",
                "Category_Display"
            ],
            as_index=False
        )["Installs_Numeric"]
        .sum()
    )


    monthly_data = monthly_data.sort_values(
        [
            "Category_Display",
            "Month"
        ]
    )


    # ========================================================
    # 14. CALCULATE MONTH-OVER-MONTH GROWTH
    # ========================================================

    monthly_data["Previous_Installs"] = (
        monthly_data
        .groupby("Category_Display")
        ["Installs_Numeric"]
        .shift(1)
    )


    monthly_data["MoM_Growth"] = (
        (
            monthly_data["Installs_Numeric"]
            -
            monthly_data["Previous_Installs"]
        )
        /
        monthly_data["Previous_Installs"]
    ) * 100


    # ========================================================
    # 15. CREATE CUMULATIVE INSTALLS
    # ========================================================

    monthly_data["Cumulative_Installs"] = (
        monthly_data
        .groupby("Category_Display")
        ["Installs_Numeric"]
        .cumsum()
    )


    # ========================================================
    # 16. CURRENT IST TIME
    # ========================================================

    IST = timezone(
        timedelta(
            hours=5,
            minutes=30
        )
    )

    current_time = datetime.now(IST)

    current_hour = current_time.hour
    current_minute = current_time.minute

    print(
        "\nCurrent IST time:",
        current_time.strftime("%I:%M %p")
    )


    # ========================================================
    # 17. TIME RESTRICTION
    # ONLY DISPLAY BETWEEN 4 PM AND 6 PM IST
    # ========================================================

    current_minutes = (
        current_hour * 60
        + current_minute
    )

    start_minutes = 16 * 60
    end_minutes = 18 * 60


    if not (
        start_minutes
        <= current_minutes
        <= end_minutes
    ):

        print("\n" + "=" * 60)
        print("CHART NOT DISPLAYED")
        print("=" * 60)
        print(
            "This visualization is available only"
            " between 4 PM and 6 PM IST."
        )
        print(
            "Current IST time:",
            current_time.strftime("%I:%M %p")
        )
        print("=" * 60)


    else:

        # ====================================================
        # 18. CREATE STACKED AREA CHART
        # ====================================================

        fig = go.Figure()


        # Get categories
        categories = (
            monthly_data["Category_Display"]
            .dropna()
            .unique()
        )


        # ====================================================
        # 19. ADD AREA FOR EACH CATEGORY
        # ====================================================

        for category in categories:

            data = monthly_data[
                monthly_data["Category_Display"]
                == category
            ].sort_values("Month")


            fig.add_trace(
                go.Scatter(
                    x=data["Month"],
                    y=data["Cumulative_Installs"],
                    mode="lines",
                    name=str(category),
                    stackgroup="one",

                    hovertemplate=
                    "<b>Category:</b> %{fullData.name}<br>"
                    "<b>Month:</b> %{x|%b %Y}<br>"
                    "<b>Cumulative Installs:</b> "
                    "%{y:,.0f}"
                    "<extra></extra>"
                )
            )


        # ====================================================
        # 20. HIGHLIGHT >25% GROWTH
        # ====================================================

        for category in categories:

            data = monthly_data[
                monthly_data["Category_Display"]
                == category
            ].sort_values("Month")


            growth_months = data[
                data["MoM_Growth"] > 25
            ]


            # Add vertical highlight lines
            for _, row in growth_months.iterrows():

                fig.add_vline(
                    x=row["Month"],
                    line_width=3,
                    line_dash="dot",
                    annotation_text=">25% Growth",
                    annotation_position="top"
                )


        # ====================================================
        # 21. CHART TITLE AND DESIGN
        # ====================================================

        fig.update_layout(

            title={
                "text":
                "Cumulative Installs by App Category<br>"
                "<sup>Highlighted months = "
                "More than 25% month-over-month growth</sup>",
                "x": 0.5
            },

            xaxis_title="Month",

            yaxis_title="Cumulative Number of Installs",

            xaxis=dict(
                tickformat="%b %Y",
                rangeslider=dict(
                    visible=True
                )
            ),

            yaxis=dict(
                tickformat=","
            ),

            hovermode="x unified",

            template="plotly_white",

            height=700,

            legend_title="Category"
        )


        # ====================================================
        # 22. DISPLAY GRAPH
        # ====================================================

        fig.show()


        # ====================================================
        # 23. SAVE GRAPH
        # ========================================================

        output_file = (
            "play_store_stacked_area_chart.html"
        )

        fig.write_html(
            output_file
        )


        # ====================================================
        # 24. DISPLAY RESULTS
        # ====================================================

        print("\n" + "=" * 60)
        print("STACKED AREA CHART CREATED SUCCESSFULLY!")
        print("=" * 60)

        print(
            "\nFiltered rows:",
            len(df)
        )

        print(
            "\nCategories included:"
        )

        print(
            df["Category_Display"]
            .value_counts()
        )

        print(
            "\nGraph saved as:",
            output_file
        )

        print(
            "\nMonthly data:"
        )

        print(
            monthly_data.head(20)
        )

CSV loaded successfully!
Total rows: 10841
Columns: ['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']

Current IST time: 07:14 PM

CHART NOT DISPLAYED
This visualization is available only between 4 PM and 6 PM IST.
Current IST time: 07:14 PM
